In [ ]:
import pandas as pd
import requests, zipfile, io
from scipy.spatial import cKDTree

GTFS_URL = "https://data.calgary.ca/download/npk7-z3bj/application%2Fx-zip-compressed"

response = requests.get(GTFS_URL)
z = zipfile.ZipFile(io.BytesIO(response.content))
trips = pd.read_csv(z.open('trips.txt'))
stops = pd.read_csv(z.open('stops.txt'))
stop_times = pd.read_csv(z.open('stop_times.txt'))

trip_to_stops = stop_times.groupby(stop_times['trip_id'].astype(str))['stop_id'].apply(list).to_dict()

def nearest_stop(lat, lon, trip_id, stops, trip_to_stoptimes):
    valid_stop_ids = trip_to_stoptimes.get(str(trip_id))
    if not valid_stop_ids:
        return None
    valid_stop_ids = [str(s) for s in valid_stop_ids]
    valid_stops = stops[stops['stop_id'].astype(str).isin(valid_stop_ids)]
    if valid_stops.empty:
        return None
    stop_coords = valid_stops[['stop_lat', 'stop_lon']].values
    tree = cKDTree(stop_coords)
    distance, index = tree.query([lat, lon])
    nearest_stop_id = valid_stops['stop_id'].iloc[index]
    return str(nearest_stop_id) if distance < 0.01 else None


test_trip_id = "74492517"
test_lat, test_lon = 51.0736, -114.063
test_stop_id = trip_to_stops.get(test_trip_id, [None])[0]  # Get the first stop for this trip

result = nearest_stop(test_lat, test_lon, test_trip_id, stops, trip_to_stops)
print("Matched stop:", result)
print("Is valid for this trip:", result in [str(s) for s in trip_to_stops.get(test_trip_id, [])])
print("Expected stop:", test_stop_id)

print("74492517" in trip_to_stops)


Matched stop: 8743
Is valid for this trip: True
Expected stop: 5160
True
